# Lab 0A Assignment: Verify Your Environment With Code

Use this notebook after you finish `03_environment_check.ipynb`.

This assignment asks you to run and complete a few short code tasks so you can prove your local setup is working.

You will:
- locate the repo root in code
- load `.env` in code
- contact the configured Ollama endpoint in code
- confirm the default model is available
- build a short environment summary in Python

In [ ]:
import json
from pathlib import Path

import requests
from dotenv import dotenv_values

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.env.example').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not find the repo root.')

repo_root = find_repo_root(Path.cwd().resolve())
print('Detected repo root:', repo_root)

## Task 1

Fill in the three paths below, then run the cell.

Expected result:
- `.env` exists
- `requirements.txt` exists
- `src/` exists

In [ ]:
# TODO: replace the None values below.
env_path = repo_root / '.env'
requirements_path = repo_root / 'requirements.txt'
src_path = repo_root / 'src'

print('.env exists:', env_path.exists())
print('requirements.txt exists:', requirements_path.exists())
print('src exists:', src_path.exists())

## Task 2

Load `.env` and print the configured model and endpoint.

In [ ]:
config = dotenv_values(env_path)
model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

print('MODEL:', model)
print('OLLAMA_BASE_URL:', ollama_base_url)

if not model or not ollama_base_url:
    raise ValueError('MODEL or OLLAMA_BASE_URL is missing from .env')

## Task 3

Build the `/api/tags` URL from `OLLAMA_BASE_URL`, send the request, and list the available models.

In [ ]:
api_root = ollama_base_url.rstrip('/')
if api_root.endswith('/v1'):
    tags_url = api_root[:-3] + '/api/tags'
else:
    tags_url = api_root + '/api/tags'

response = requests.get(tags_url, timeout=10)
response.raise_for_status()
available_models = [item.get('name') for item in response.json().get('models', []) if item.get('name')]

print('Tags URL:', tags_url)
print('Available models:')
for model_name in available_models:
    print('-', model_name)

## Task 4

Check whether the default model from `.env` is available at the configured endpoint.

In [ ]:
model_available = model in available_models

print('Default model available:', model_available)

if not model_available:
    raise ValueError(f'The model {model!r} is not available at the configured endpoint.')

## Task 5

Complete the summary dictionary below, then print it as formatted JSON.

For `repo_root_name`, use only the folder name, not the full absolute path.

In [ ]:
# TODO: complete the dictionary values below.
environment_summary = {
    'repo_root_name': repo_root.name,
    'default_model': model,
    'ollama_base_url': ollama_base_url,
    'available_model_count': len(available_models),
    'default_model_available': model_available,
}

print(json.dumps(environment_summary, indent=2))

## Submission

Save the notebook with:
- the completed code cells
- the printed model list
- the final `environment_summary` output